In [1]:
%pip install scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [2]:
# =========================
# 0. INSTALL & IMPORTS
# =========================
# !pip install google-cloud-storage pymupdf torchvision torch sklearn matplotlib pandas tqdm

import os
from pathlib import Path
import json

import fitz  # PyMuPDF
from PIL import Image

import pandas as pd
import numpy as np
from tqdm import tqdm

from google.cloud import storage

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score, confusion_matrix
import matplotlib.pyplot as plt


c:\Users\18vic\anaconda3\envs\myenv\lib\site-packages\google\api_core\_python_version_support.py:266: FutureWarning: You are using a Python version (3.10.18) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


In [3]:
# =========================
# 1. CONFIG & GCP CLIENT
# =========================

# IMPORTANT: do NOT hardcode your private key.
# Set this env var to point to the JSON you already have locally.
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = r"../data\turing-agent-358210-a38a4820a9ce.json"

BUCKET_NAME = "capstone-ii-applicant-documents"
PDF_PREFIX = "applicant-documents-pdf/"
META_PREFIX = "applicant-metadata/"   # adjust if different
IMAGE_ROOT = Path("data/images")
IMAGE_ROOT.mkdir(parents=True, exist_ok=True)

storage_client = storage.Client()
bucket = storage_client.bucket(BUCKET_NAME)


In [4]:
# =========================
# 2. LABEL EXTRACTION
# =========================

def extract_doc_labels_for_applicant(app_record):
    """
    app_record: one JSON document like the example you sent.
    
    Rule (you can adjust):
      - Label = 1 ("real") if:
          documentInGovernmentWebsite == 1 AND
          ocrMatchesApplication == 1 AND
          no fraudCheck for that doc has status == "fail"
      - Otherwise label = 0 ("fraud/suspicious")
    """
    doc_labels = {}
    
    # Group fraudChecks by doc name
    checks_by_doc = {}
    for fc in app_record.get("fraudChecks", []):
        label_str = fc["label"]
        doc_name = label_str.split(" - ", 1)[0]  # "paystub.pdf"
        checks_by_doc.setdefault(doc_name, []).append(fc)
    
    for doc in app_record.get("documents", []):
        doc_name = doc["name"]
        gov_ok = doc.get("documentInGovernmentWebsite", 0) == 1
        ocr_ok = doc.get("ocrMatchesApplication", 0) == 1
        doc_checks = checks_by_doc.get(doc_name, [])
        has_fail = any(c["status"].lower() == "fail" for c in doc_checks)
        
        if gov_ok and ocr_ok and not has_fail:
            label = 1
        else:
            label = 0
        
        doc_labels[doc_name] = label
    
    return doc_labels


def load_metadata_records_from_gcs():
    """
    Reads all JSON metadata files from META_PREFIX and returns a list of dicts.
    Assumes each file is a single applicant record.
    """
    records = []
    blobs = bucket.list_blobs(prefix=META_PREFIX)
    for blob in blobs:
        if not blob.name.endswith(".json"):
            continue
        content = blob.download_as_text()
        rec = json.loads(content)
        records.append(rec)
    return records


def build_label_mapping(metadata_records):
    """
    metadata_records = [ [ applicant_dict1, applicant_dict2, ... ] ]
    So we must index metadata_records[0] to get the list of applicants.
    """
    applicants = metadata_records[0]       # <- FIX
    mapping = {}

    for applicant in applicants:
        applicant_id = applicant["applicantFolder"]  # e.g., 'applicant_1'

        for doc in applicant["documents"]:
            doc_name = doc["name"]  # e.g., 'Sample 4.pdf'

            # Label = fully verified AND matches OCR
            gov = doc.get("documentInGovernmentWebsite", 0)
            ocr = doc.get("ocrMatchesApplication", 0)

            label = 1 if (gov == 1 and ocr == 1) else 0

            mapping[(applicant_id, doc_name)] = label

    return mapping


In [5]:
# metadata_records = load_metadata_records_from_gcs()

# print("Type of metadata_records:", type(metadata_records))
# print("Length:", len(metadata_records))

# print("\nFirst record raw:")
# print(metadata_records[0])

# # If record contains nested documents field, inspect one doc too
# if isinstance(metadata_records[0], dict):
#     print("\nKeys in first record:", metadata_records[0].keys())
#     if "documents" in metadata_records[0]:
#         print("\nFirst document entry:")
#         print(metadata_records[0]["documents"][0])




In [6]:
# =========================
# 3. PDF -> IMAGE DATASET
# =========================

def pdf_bytes_to_images_force(pdf_bytes: bytes, dpi: int = 300):
    """
    Rasterize every PDF page to an image.
    Handles vector-only PDFs by forcing a pixel rendering pipeline.
    """
    images = []
    try:
        with fitz.open(stream=pdf_bytes, filetype="pdf") as doc:
            zoom = dpi / 72.0
            mat = fitz.Matrix(zoom, zoom)

            for i in range(len(doc)):
                page = doc.load_page(i)
                pix = page.get_pixmap(matrix=mat, alpha=False)  # force pixel output
                img = Image.frombytes("RGB", (pix.width, pix.height), pix.samples)
                images.append(img)
    except Exception as e:
        print(f"PDF conversion failed: {e}")
    return images

def download_pdf_bytes(blob):
    raw = blob.download_as_bytes()

    try:
        # Attempt to open as JSON
        import json
        data = json.loads(raw)

        # Case: Node.js Buffer structure
        if isinstance(data, dict) and "file" in data and "data" in data["file"]:
            pdf_bytes = bytes(data["file"]["data"])
            return pdf_bytes

        # Case: Already base64 or nested under "data"
        if "data" in data and isinstance(data["data"], list):
            pdf_bytes = bytes(data["data"])
            return pdf_bytes

        # Fallback: Raw PDF bytes?
        return raw

    except Exception:
        # If not JSON — assume real PDF
        return raw

def build_image_dataset_from_gcs_force(dpi=300, max_docs=None):
    """
    Converts ALL PDFs to images by forcing raster rendering.
    """
    records = []
    blobs = bucket.list_blobs(prefix=PDF_PREFIX)

    for idx, blob in enumerate(tqdm(blobs, desc="Rendering PDFs")):
        if not blob.name.lower().endswith(".pdf"):
            continue

        if max_docs and idx >= max_docs:
            break

        pdf_bytes = download_pdf_bytes(blob)

        parts = blob.name.split("/")
        if len(parts) < 3:
            continue

        applicant_id = parts[-2]
        doc_id = parts[-1]

        images = pdf_bytes_to_images_force(pdf_bytes, dpi=dpi)
        if not images:
            print(f"SKIPPED (no pages rendered): {blob.name}")
            continue

        out_dir = IMAGE_ROOT / applicant_id
        out_dir.mkdir(parents=True, exist_ok=True)

        for page_idx, img in enumerate(images):
            img_filename = f"{Path(doc_id).stem}_page{page_idx}.png"
            img_path = out_dir / img_filename
            img.save(img_path)

            records.append({
                "applicant_id": applicant_id,
                "doc_id": doc_id,
                "page_idx": page_idx,
                "image_path": str(img_path)
            })

    df = pd.DataFrame(records)
    return df




# Build label mapping and dataset
metadata_records = load_metadata_records_from_gcs()
label_mapping = build_label_mapping(metadata_records)
print(len(label_mapping), "document labels loaded.")
df = build_image_dataset_from_gcs_force(label_mapping)
print(df.head(), "\nTotal images:", len(df))


4 document labels loaded.


Rendering PDFs: 3it [00:00,  3.56it/s]

PDF conversion failed: unsupported operand type(s) for /: 'dict' and 'float'
SKIPPED (no pages rendered): applicant-documents-pdf/applicant_1/APPLICATION FORM_46.pdf
PDF conversion failed: unsupported operand type(s) for /: 'dict' and 'float'
SKIPPED (no pages rendered): applicant-documents-pdf/applicant_1/DEVIATION APPROVAL MAILS_34.pdf


Rendering PDFs: 5it [00:01,  6.02it/s]

PDF conversion failed: unsupported operand type(s) for /: 'dict' and 'float'
SKIPPED (no pages rendered): applicant-documents-pdf/applicant_1/DISBURSEMENT MEMO_43.pdf
PDF conversion failed: unsupported operand type(s) for /: 'dict' and 'float'
SKIPPED (no pages rendered): applicant-documents-pdf/applicant_1/DISBURSEMENT MEMO_48.pdf
PDF conversion failed: unsupported operand type(s) for /: 'dict' and 'float'
SKIPPED (no pages rendered): applicant-documents-pdf/applicant_1/DISBURSEMENT MEMO_6.pdf


...

PDF conversion failed: unsupported operand type(s) for /: 'dict' and 'float'
SKIPPED (no pages rendered): applicant-documents-pdf/applicant_4/DISBURSEMENT MEMO_39.pdf
PDF conversion failed: unsupported operand type(s) for /: 'dict' and 'float'
SKIPPED (no pages rendered): applicant-documents-pdf/applicant_4/DISBURSEMENT MEMO_44.pdf


Rendering PDFs: 82it [00:14,  9.07it/s]

PDF conversion failed: unsupported operand type(s) for /: 'dict' and 'float'
SKIPPED (no pages rendered): applicant-documents-pdf/applicant_4/DISBURSEMENT REQUEST FORM_20.pdf
PDF conversion failed: unsupported operand type(s) for /: 'dict' and 'float'
SKIPPED (no pages rendered): applicant-documents-pdf/applicant_4/DISBURSEMENT REQUEST FORM_37.pdf
PDF conversion failed: unsupported operand type(s) for /: 'dict' and 'float'
SKIPPED (no pages rendered): applicant-documents-pdf/applicant_4/DISBURSEMENT REQUEST FORM_8.pdf


Rendering PDFs: 83it [00:14,  5.63it/s]

PDF conversion failed: unsupported operand type(s) for /: 'dict' and 'float'
SKIPPED (no pages rendered): applicant-documents-pdf/applicant_4/Question 17.pdf
Empty DataFrame
Columns: []
Index: [] 
Total images: 0


In [7]:
for blob in bucket.list_blobs(prefix=PDF_PREFIX):
    print("TYPE PDF BYTES:", type(pdf_bytes), blob.name)

NameError: name 'pdf_bytes' is not defined

In [ ]:
# =========================
# 4. DATASET & DATALOADERS
# =========================

def pad_to_square(img: Image.Image, fill=255) -> Image.Image:
    w, h = img.size
    if w == h:
        return img
    if w > h:
        pad_top = (w - h) // 2
        pad_bottom = w - h - pad_top
        padding = (0, pad_top, 0, pad_bottom)
    else:
        pad_left = (h - w) // 2
        pad_right = h - w - pad_left
        padding = (pad_left, 0, pad_right, 0)
    return ImageOps.expand(img, border=padding, fill=fill)

from PIL import ImageOps  # used in pad_to_square

# Basic transforms (you can extend with augmentation if you want)
train_transform = transforms.Compose([
    transforms.Lambda(lambda im: pad_to_square(im)),
    transforms.Resize((512, 512)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5],
                         std=[0.5, 0.5, 0.5]),
])

val_transform = transforms.Compose([
    transforms.Lambda(lambda im: pad_to_square(im)),
    transforms.Resize((512, 512)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5],
                         std=[0.5, 0.5, 0.5]),
])


class FraudDocsDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["image_path"]).convert("RGB")
        if self.transform:
            img = self.transform(img)
        label = torch.tensor(row["label"], dtype=torch.float32)
        return img, label


# Train / val / test split
train_df, temp_df = train_test_split(
    df,
    test_size=0.3,
    stratify=df["label"],
    random_state=42
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    stratify=temp_df["label"],
    random_state=42
)

train_ds = FraudDocsDataset(train_df, transform=train_transform)
val_ds   = FraudDocsDataset(val_df, transform=val_transform)
test_ds  = FraudDocsDataset(test_df, transform=val_transform)

train_loader = DataLoader(train_ds, batch_size=8, shuffle=True, num_workers=2)
val_loader   = DataLoader(val_ds, batch_size=8, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_ds, batch_size=8, shuffle=False, num_workers=2)

len(train_ds), len(val_ds), len(test_ds)


(9, 2, 2)

In [ ]:
# # =========================
# # 5. MODEL Instantiation
# # =========================

import torch
from pathlib import Path
from models.mantranet.MantraNet.mantranet import MantraNet

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ROOT = Path.cwd().parent
MODEL_DIR = ROOT / "models" / "mantranet" / "MantraNet"

IMTFE_WEIGHTS = MODEL_DIR / "IMTFEv4.pt"
ANOMALY_WEIGHTS = MODEL_DIR / "AnomalyDetectorv4.pt"
MANTRANET_WEIGHTS = MODEL_DIR / "MantraNetv4.pt"

print("Loading weights from:")
print(" -", IMTFE_WEIGHTS)
print(" -", ANOMALY_WEIGHTS)
print(" -", MANTRANET_WEIGHTS)

model = MantraNet(device=device)

# Load pretrained modules
IMTFE_state = torch.load(IMTFE_WEIGHTS, map_location=device)
ANO_state = torch.load(ANOMALY_WEIGHTS, map_location=device)
MAIN_state = torch.load(MANTRANET_WEIGHTS, map_location=device)

model.IMTFE.load_state_dict({k.replace("module.", ""): v for k,v in IMTFE_state.items()})
model.AnomalyDetector.load_state_dict({k.replace("module.", ""): v for k,v in ANO_state.items()})
model.load_state_dict({k.replace("module.", ""): v for k,v in MAIN_state.items()}, strict=False)

model.to(device)
model.eval()

print("✓ ManTraNet successfully initialized")


Loading weights from:
 - c:\Users\18vic\OneDrive\Desktop\capstone\capstone\models\mantranet\MantraNet\IMTFEv4.pt
 - c:\Users\18vic\OneDrive\Desktop\capstone\capstone\models\mantranet\MantraNet\AnomalyDetectorv4.pt
 - c:\Users\18vic\OneDrive\Desktop\capstone\capstone\models\mantranet\MantraNet\MantraNetv4.pt
✓ ManTraNet successfully initialized


In [ ]:

# --------------------------------------------------
# 2. Dataset with uniform resizing + metadata return
# --------------------------------------------------
from torchvision.transforms import functional as TF

TARGET_SIZE = (512, 512)  # Standardized size for ManTraNet

class FraudImageDataset(Dataset):
    def __init__(self, df):
        self.df = df

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["image_path"]).convert("RGB")

        # Resize for consistent batch tensor size
        img = TF.resize(img, TARGET_SIZE)
        img = TF.to_tensor(img)

        return (
            img.float(),
            str(row["image_path"]),
            str(row["applicant_id"]),
            str(row["doc_id"]),
            int(row["page_idx"])
        )

dataset = FraudImageDataset(df)
dataset_loader = DataLoader(dataset, batch_size=2, shuffle=False, num_workers=0)


# --------------------------------------------------
# 3. Page-level fraud scoring function
# --------------------------------------------------
def fraud_score_from_heatmap(mask: torch.Tensor):
    mask_np = mask.detach().cpu().numpy()
    mean_val = float(np.mean(mask_np))
    max_val = float(np.max(mask_np))
    return (mean_val + max_val) / 2  # balanced suspiciousness index


# --------------------------------------------------
# 4. Run full inference
# --------------------------------------------------
fraud_scores = []

for imgs, paths, applicants, docs, pages in dataset_loader:
    imgs = imgs.to(device)

    with torch.no_grad():
        heatmaps = model(imgs)  # B x 1 x H x W

    for i in range(imgs.size(0)):
        score = fraud_score_from_heatmap(heatmaps[i, 0])
        fraud_scores.append({
            "image_path": paths[i],
            "applicant_id": applicants[i],
            "doc_id": docs[i],
            "page_idx": pages[i],
            "fraud_score": score
        })

fraud_df = pd.DataFrame(fraud_scores)
fraud_df.to_csv("fraud_page_scores.csv", index=False)
print("Page score results:")
print(fraud_df.head(), "\n")


# --------------------------------------------------
# 5. Aggregate document-level and applicant-level scores
# --------------------------------------------------
doc_scores = fraud_df.groupby(["applicant_id", "doc_id"])["fraud_score"].max().reset_index()
doc_scores.to_csv("fraud_document_scores.csv", index=False)

applicant_scores = fraud_df.groupby("applicant_id")["fraud_score"].max().reset_index()
applicant_scores.to_csv("fraud_applicant_scores.csv", index=False)

print("✓ Saved outputs:")
print(" - fraud_page_scores.csv")
print(" - fraud_document_scores.csv")
print(" - fraud_applicant_scores.csv")

Page score results:
                                   image_path applicant_id        doc_id  \
0  data\images\applicant_1\Sample 4_page0.png  applicant_1  Sample 4.pdf   
1  data\images\applicant_1\Sample 4_page1.png  applicant_1  Sample 4.pdf   
2  data\images\applicant_1\Sample 4_page2.png  applicant_1  Sample 4.pdf   
3  data\images\applicant_1\Sample 4_page3.png  applicant_1  Sample 4.pdf   
4  data\images\applicant_2\Sample 4_page0.png  applicant_2  Sample 4.pdf   

    page_idx  fraud_score  
0  tensor(0)     0.513264  
1  tensor(1)     0.510485  
2  tensor(2)     0.503294  
3  tensor(3)     0.502308  
4  tensor(0)     0.513265   

✓ Saved outputs:
 - fraud_page_scores.csv
 - fraud_document_scores.csv
 - fraud_applicant_scores.csv


In [ ]:
def fraud_score_from_heatmap(mask: torch.Tensor):
    mask_np = mask.detach().cpu().numpy()
    mean_val = float(np.mean(mask_np))
    max_val = float(np.max(mask_np))
    return (mean_val + max_val) / 2


In [ ]:
fraud_df = pd.DataFrame(fraud_scores)
fraud_df.to_csv("fraud_scores.csv", index=False)
print(fraud_df.head())


In [ ]:
# =========================
# 6. EVALUATION & PLOTS
# =========================

test_loss, y_test, test_logits = run_epoch(test_loader, model, criterion, optimizer=None)
test_probs = torch.sigmoid(torch.from_numpy(test_logits)).numpy()
test_preds = (test_probs >= 0.5).astype(int)

test_acc = accuracy_score(y_test, test_preds)
try:
    test_auc = roc_auc_score(y_test, test_probs)
except ValueError:
    test_auc = float("nan")

cm = confusion_matrix(y_test, test_preds)

print(f"\nTest Loss: {test_loss:.4f} | Test Acc: {test_acc:.3f} | Test AUC: {test_auc:.3f}")
print("Confusion Matrix:\n", cm)


# Plot training vs validation loss
plt.figure()
plt.plot(range(1, EPOCHS + 1), train_losses, label="Train Loss")
plt.plot(range(1, EPOCHS + 1), val_losses, label="Val Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training vs Validation Loss")
plt.legend()
plt.show()


# Simple ROC curve (optional)
from sklearn.metrics import roc_curve

fpr, tpr, _ = roc_curve(y_test, test_probs)
plt.figure()
plt.plot(fpr, tpr, label=f"ROC (AUC={test_auc:.3f})")
plt.plot([0, 1], [0, 1], linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Test ROC Curve")
plt.legend()
plt.show()


In [ ]:
def count_docs_per_applicant(fraud_df):
    doc_counts = (
        fraud_df
        .groupby("applicant_id")["doc_id"]
        .nunique()
        .reset_index(name="num_documents")
    )
    return doc_counts

doc_counts_df = count_docs_per_applicant(fraud_df)
print(doc_counts_df)


  applicant_id  num_documents
0  applicant_1              1
1  applicant_2              1
2  applicant_3              1
3  applicant_4              1


In [ ]:
# ============================================
# 0. INSTALLS (run once in the environment)
# ============================================
# !pip install google-cloud-storage pymupdf torchvision torch sklearn matplotlib pandas tqdm opencv-python pytorch-lightning

# ============================================
# 1. IMPORTS
# ============================================
import os
from pathlib import Path
import json

import fitz  # PyMuPDF
from PIL import Image

import pandas as pd
import numpy as np
from tqdm import tqdm

from google.cloud import storage

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

# ManTraNet code from your repo
import sys
from collections import OrderedDict

# ============================================
# 2. CONFIG & GCP CLIENT
# ============================================

# Set your service account JSON (adjust path as needed)
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = r"../data\turing-agent-358210-a38a4820a9ce.json"

BUCKET_NAME = "capstone-ii-applicant-documents"
PDF_PREFIX = "applicant-documents-pdf/"
META_PREFIX = "applicant-metadata/"

# Where to store images locally
IMAGE_ROOT = Path("data/images")
IMAGE_ROOT.mkdir(parents=True, exist_ok=True)

# GCS client
storage_client = storage.Client()
bucket = storage_client.bucket(BUCKET_NAME)

print("GCP client initialized.")


# ============================================
# 3. OPTIONAL: METADATA LOADING (NOT USED IN PIPELINE, BUT AVAILABLE)
# ============================================
def load_metadata_records_from_gcs():
    """
    Loads all JSON metadata files from META_PREFIX into a list.
    """
    records = []
    blobs = bucket.list_blobs(prefix=META_PREFIX)
    for blob in blobs:
        if not blob.name.lower().endswith(".json"):
            continue
        content = blob.download_as_text()
        try:
            data = json.loads(content)
            # Some JSONs might contain a list at top-level
            if isinstance(data, list):
                records.extend(data)
            else:
                records.append(data)
        except Exception as e:
            print(f"Error reading metadata {blob.name}: {e}")
    return records


def build_label_mapping(metadata_records):
    """
    Example label mapping if you later want supervision.
    Currently NOT used in ManTraNet scoring.
    Expected structure per metadata record:
      {
        "id": "...",
        "applicantFolder": "applicant_1",
        "documents": [
           {
             "name": "Sample 4.pdf",
             "documentInGovernmentWebsite": 1,  # or 0
             ...
           },
           ...
        ]
      }
    Returns: dict[(applicantFolder, doc_name)] -> label (0/1)
    """
    mapping = {}

    for rec_idx, rec in enumerate(metadata_records):
        if not isinstance(rec, dict):
            continue

        applicant_id = rec.get("applicantFolder") or f"applicant_{rec.get('applicantNumber', rec_idx+1)}"
        docs = rec.get("documents", [])

        for d in docs:
            name = d.get("name")
            if not name:
                continue

            gov = d.get("documentInGovernmentWebsite")
            if gov is None:
                # If no label field, skip or assign default
                continue

            label = int(gov)
            mapping[(applicant_id, name)] = label

    return mapping


# If you want metadata for later, you can uncomment this:
# metadata_records = load_metadata_records_from_gcs()
# label_mapping = build_label_mapping(metadata_records)
# print(len(label_mapping), "document labels loaded.")


# ============================================
# 4. PDF → IMAGES (FORCE RASTERIZATION)
# ============================================
def pdf_bytes_to_images_force(pdf_bytes: bytes, dpi: int = 300):
    """
    Rasterize every PDF page to an image.
    Handles vector-only PDFs by forcing a pixel rendering pipeline.
    """
    images = []
    try:
        with fitz.open(stream=pdf_bytes, filetype="pdf") as doc:
            zoom = dpi / 72.0  # 72 dpi is PDF default
            mat = fitz.Matrix(zoom, zoom)

            for i in range(len(doc)):
                page = doc.load_page(i)
                # Force raster render at given zoom
                pix = page.get_pixmap(matrix=mat, alpha=False)
                img = Image.frombytes("RGB", (pix.width, pix.height), pix.samples)
                images.append(img)
    except Exception as e:
        print(f"PDF conversion failed: {e}")
    return images


def build_image_dataset_from_gcs_force(dpi=300, max_docs=None):
    """
    Converts ALL PDFs to images by forcing raster rendering.
    Returns a DataFrame:
       [applicant_id, doc_id, page_idx, image_path]
    """
    records = []
    blobs = bucket.list_blobs(prefix=PDF_PREFIX)

    for idx, blob in enumerate(tqdm(blobs, desc="Rendering PDFs")):
        name_lower = blob.name.lower()

        if not name_lower.endswith(".pdf"):
            continue  # skip folders / non-pdf

        if max_docs is not None and idx >= max_docs:
            break

        # Download raw PDF bytes
        pdf_bytes = blob.download_as_bytes()

        parts = blob.name.split("/")
        if len(parts) < 3:
            # Something like "applicant-documents-pdf/"
            continue

        applicant_id = parts[-2]  # e.g. "applicant_1"
        doc_id = parts[-1]        # e.g. "Sample 4.pdf"

        images = pdf_bytes_to_images_force(pdf_bytes, dpi=dpi)
        if not images:
            print(f"SKIPPED (no pages rendered): {blob.name}")
            continue

        out_dir = IMAGE_ROOT / applicant_id
        out_dir.mkdir(parents=True, exist_ok=True)

        for page_idx, img in enumerate(images):
            img_filename = f"{Path(doc_id).stem}_page{page_idx}.png"
            img_path = out_dir / img_filename
            img.save(img_path)

            records.append({
                "applicant_id": applicant_id,
                "doc_id": doc_id,
                "page_idx": page_idx,
                "image_path": str(img_path)
            })

    df = pd.DataFrame(records)
    return df


print("Starting PDF → Image conversion...")
df = build_image_dataset_from_gcs_force(dpi=300)   # <-- NO label_mapping here
print(df.head(), "\nTotal images:", len(df))

# Quick sanity check per applicant
print("\nDocs per applicant:")
if not df.empty:
    print(df.groupby("applicant_id")["doc_id"].nunique())


# ============================================
# 5. LOAD MANTRANET (PRETRAINED)
# ============================================
# Ensure project root is on sys.path so we can import your local module
print("\nSetting up ManTraNet…")
CWD = Path.cwd()          # notebooks/
ROOT = CWD.parent         # capstone/
print("CWD:", CWD)
print("ROOT:", ROOT)

sys.path.append(str(ROOT))

from models.mantranet.MantraNet.mantranet import MantraNet

if torch.cuda.is_available():
    print("Using CUDA!")
    device = torch.device("cuda")
else:
    print("No CUDA detected, CPU inference will be extremely slow.")
    device = torch.device("cpu")


MODEL_DIR = ROOT / "models" / "mantranet" / "MantraNet"
IMTFE_WEIGHTS = MODEL_DIR / "IMTFEv4.pt"
ANOMALY_WEIGHTS = MODEL_DIR / "AnomalyDetectorv4.pt"
MANTRANET_WEIGHTS = MODEL_DIR / "MantraNetv4.pt"

print("Weight paths:")
print("  IMTFE:", IMTFE_WEIGHTS)
print("  AnomalyDetector:", ANOMALY_WEIGHTS)
print("  ManTraNet:", MANTRANET_WEIGHTS)

# Instantiate & load weights
model = MantraNet(device=device)

# Some checkpoints are saved with "module." prefix from DataParallel
def strip_module_prefix(state_dict):
    return {k.replace("module.", ""): v for k, v in state_dict.items()}

IMTFE_state = torch.load(IMTFE_WEIGHTS, map_location=device)
ANO_state = torch.load(ANOMALY_WEIGHTS, map_location=device)
MAIN_state = torch.load(MANTRANET_WEIGHTS, map_location=device)

model.IMTFE.load_state_dict(strip_module_prefix(IMTFE_state))
model.AnomalyDetector.load_state_dict(strip_module_prefix(ANO_state))
model.load_state_dict(strip_module_prefix(MAIN_state), strict=False)

model.to(device)
model.eval()
print("✓ ManTraNet loaded successfully on", device)


# ============================================
# 6. DATASET & DATALOADER FOR MANTRANET
# ============================================
# ManTraNet expects input range roughly [0, 255], and does its own normalization.
# So we convert PIL image → uint8 numpy → torch float [C,H,W] with values 0–255.

class FraudImageDataset(Dataset):
    def __init__(self, df: pd.DataFrame):
        self.df = df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["image_path"]).convert("RGB")

        # To tensor with 0–255 scaling
        arr = np.array(img)              # H,W,3 uint8
        tensor = torch.from_numpy(arr)   # H,W,3
        tensor = tensor.permute(2, 0, 1) # 3,H,W
        tensor = tensor.float()          # keep 0–255

        return (
            tensor,
            str(row["image_path"]),
            str(row["applicant_id"]),
            str(row["doc_id"]),
            int(row["page_idx"])
        )


dataset = FraudImageDataset(df)

# batch_size=1 to avoid size mismatch issues across pages with different resolutions
dataset_loader = DataLoader(dataset, batch_size=1, shuffle=False, num_workers=0)

print("Total samples for scoring:", len(dataset))


# ============================================
# 7. FRAUD SCORE FROM HEATMAP
# ============================================
def fraud_score_from_heatmap(mask: torch.Tensor) -> float:
    """
    Simple scalar score from ManTraNet heatmap:
    average of mean and max anomaly intensity.
    """
    # mask: [H,W] or [1,H,W]
    if mask.dim() == 3 and mask.size(0) == 1:
        mask = mask[0]
    mask_np = mask.detach().cpu().numpy()
    mean_val = float(np.mean(mask_np))
    max_val = float(np.max(mask_np))
    return (mean_val + max_val) / 2.0


# ============================================
# 8. INFERENCE LOOP (PAGE-LEVEL SCORES)
# ============================================
fraud_scores = []

print("\nRunning ManTraNet inference on all pages...")
with torch.no_grad():
    for imgs, paths, applicants, docs, pages in tqdm(dataset_loader, desc="Scoring pages"):
        # imgs: [B,3,H,W], but B=1
        imgs = imgs.to(device)

        heatmaps = model(imgs)  # expect [B,1,H,W]
        # Make sure output shape is as expected
        if isinstance(heatmaps, (list, tuple)):
            # If the model returns something nested, adapt here
            heatmaps = heatmaps[0]

        for i in range(imgs.size(0)):
            mask = heatmaps[i, 0]  # [H,W]
            score = fraud_score_from_heatmap(mask)

            fraud_scores.append({
                "image_path": paths[i],
                "applicant_id": applicants[i],
                "doc_id": docs[i],
                "page_idx": int(pages[i]),
                "fraud_score": score
            })

fraud_df = pd.DataFrame(fraud_scores)
fraud_df.to_csv("fraud_page_scores.csv", index=False)

print("\nPage Score Sample:")
print(fraud_df.head())


# ============================================
# 9. AGGREGATION: DOC-LEVEL & APPLICANT-LEVEL
# ============================================
if fraud_df.empty:
    print("\nWARNING: No fraud scores produced. Check PDF → image step.")
else:
    # Document-level: max page score per doc
    doc_scores = (
        fraud_df
        .groupby(["applicant_id", "doc_id"])["fraud_score"]
        .max()
        .reset_index()
    )
    doc_scores.to_csv("fraud_document_scores.csv", index=False)

    # Applicant-level: max document score per applicant
    applicant_scores = (
        doc_scores
        .groupby("applicant_id")["fraud_score"]
        .max()
        .reset_index()
    )
    applicant_scores.to_csv("fraud_applicant_scores.csv", index=False)

    print("\nDoc-level score sample:")
    print(doc_scores.head())

    print("\nApplicant-level score sample:")
    print(applicant_scores.head())

    print("\nFiles saved:")
    print(" - fraud_page_scores.csv")
    print(" - fraud_document_scores.csv")
    print(" - fraud_applicant_scores.csv")


c:\Users\18vic\anaconda3\envs\myenv\lib\site-packages\google\api_core\_python_version_support.py:266: FutureWarning: You are using a Python version (3.10.18) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


GCP client initialized.
Starting PDF → Image conversion...


Rendering PDFs: 85it [02:36,  1.84s/it]


  applicant_id                   doc_id  page_idx  \
0  applicant_1  APPLICATION FORM_46.pdf         0   
1  applicant_1  APPLICATION FORM_46.pdf         1   
2  applicant_1  APPLICATION FORM_46.pdf         2   
3  applicant_1  APPLICATION FORM_46.pdf         3   
4  applicant_1  APPLICATION FORM_46.pdf         4   

                                          image_path  
0  data\images\applicant_1\APPLICATION FORM_46_pa...  
1  data\images\applicant_1\APPLICATION FORM_46_pa...  
2  data\images\applicant_1\APPLICATION FORM_46_pa...  
3  data\images\applicant_1\APPLICATION FORM_46_pa...  
4  data\images\applicant_1\APPLICATION FORM_46_pa...   
Total images: 271

Docs per applicant:
applicant_id
applicant_1    31
applicant_2    20
applicant_3    23
applicant_4     8
Name: doc_id, dtype: int64

Setting up ManTraNet…
CWD: c:\Users\18vic\OneDrive\Desktop\capstone\capstone\notebooks
ROOT: c:\Users\18vic\OneDrive\Desktop\capstone\capstone
No CUDA detected, CPU inference will be extremely slow.

Scoring pages:   0%|          | 0/271 [00:00<?, ?it/s]